In [ ]:
import glob
import pandas as pd

# 원본 파일 목록 (최종본을 data/ 로 옮겼으므로 이제 16개만 잡힌다)
files = sorted(glob.glob("data/population/*.csv"))
print("파일 수:", len(files), "(16이어야 정상)")

frames = []
for f in files:
    # encoding="cp949" : euc-kr보다 넓은 범위를 커버해서 한글 csv에 더 안전하다
    # thousands=","    : "1,390,658" 처럼 쉼표 박힌 숫자를 자동으로 int로 바꿔 준다
    #                    (이걸 안 쓰면 전부 문자열로 읽혀서 나중에 일일이 replace 해야 함)
    df = pd.read_csv(f, encoding="cp949", thousands=",")
    frames.append(df)

# 16개 시도 파일을 세로로 이어 붙인다 (컬럼 구조는 16개 파일 모두 동일)
raw = pd.concat(frames, ignore_index=True)

print("병합 결과:", raw.shape, "(285행 × 235열이어야 정상)")
print("지역 중복:", raw["행정구역"].duplicated().sum(), "건")
print()
print("숫자 타입 확인:", raw["2026년01월_계_총인구수"].dtype, "(int64면 성공)")
print(raw["행정구역"].head(3).tolist())


In [ ]:
import re

def split_region(s):
    """'광주광역시 동구 (2911000000)' → ('광주광역시 동구', '2911000000')"""
    # ^(.*?)  : 앞부분을 최소한으로 잡되   → 지역명
    # \s*     : 이름과 괄호 사이 공백 무시 (원본에 공백이 1~2개로 들쭉날쭉함)
    # \((\d+)\)$ : 맨 끝 괄호 안 숫자      → 행정구역코드
    m = re.match(r"^(.*?)\s*\((\d+)\)$", s.strip())
    if m:
        return m.group(1).strip(), m.group(2)
    return s.strip(), None      # 괄호 형식이 아닌 예외 행 대비

# apply 결과를 pd.Series로 반환하면 두 컬럼으로 한 번에 펼칠 수 있다
raw[["행정구역명", "행정구역코드"]] = raw["행정구역"].apply(
    lambda s: pd.Series(split_region(s))
)

# 코드 분리가 실패한 행이 있는지 확인 (0이어야 정상)
print("코드 추출 실패:", raw["행정구역코드"].isna().sum(), "건")
print()
print(raw[["행정구역", "행정구역명", "행정구역코드"]].head(5).to_string())


In [ ]:
# 1) melt 대상 컬럼 고르기
#    - "_남_", "_여_" 만 → '계'는 남+여 합계라 같이 넣으면 인구가 두 배가 된다
#    - '총인구수', '연령구간인구수'는 합계 컬럼이라 제외 (역시 중복)
age_cols = [c for c in raw.columns
            if ("_남_" in c or "_여_" in c)
            and "총인구수" not in c
            and "연령구간인구수" not in c]

print("melt 대상 컬럼:", len(age_cols), "개 (6개월 × 2성별 × 11구간 = 132)")

# 2) 가로 → 세로
#    id_vars    : 그대로 남길 식별 컬럼
#    value_vars : 세로로 접을 컬럼들
#    → 컬럼명은 '항목', 값은 '인구'로 들어간다
long = raw.melt(id_vars=["행정구역명", "행정구역코드"],
                value_vars=age_cols,
                var_name="항목",
                value_name="인구")

print("변환 결과:", long.shape, "(285행 × 132컬럼 = 37,620행)")

# 3) '2026년01월_남_20~29세' 를 세 조각으로 분해
#    n=2 : 앞에서 2번만 자른다 → '100세 이상'처럼 뒤에 공백이 있어도 안 깨진다
parts = long["항목"].str.split("_", n=2, expand=True)

long["STRD_YYMM"] = parts[0].str.replace("년", "").str.replace("월", "")  # 2026년01월 → 202601
long["성별"] = parts[1]      # 남 / 여
long["연령10"] = parts[2]    # 0~9세 … 100세 이상

print()
print(long[["항목", "STRD_YYMM", "성별", "연령10", "인구"]].head(3).to_string())
print()
print("성별 값:", long["성별"].unique())
print("연령 구간:", long["연령10"].nunique(), "종류")


In [ ]:
# 1) 10세 단위 11구간 → BC의 6구간으로 매핑
#    딕셔너리에 없는 값(60~69세 이상 전부)은 fillna로 '60대이상' 처리 → 규칙이 짧아진다
연령매핑 = {
    "0~9세": "20대이하", "10~19세": "20대이하",   # 미성년까지 한 덩어리
    "20~29세": "20대",
    "30~39세": "30대",
    "40~49세": "40대",
    "50~59세": "50대",
    # 60~69세 / 70~79세 / 80~89세 / 90~99세 / 100세 이상 → 아래 fillna에서 '60대이상'
}
long["연령대"] = long["연령10"].map(연령매핑).fillna("60대이상")

# 2) BC 데이터와 바로 조인할 수 있게 코드값으로 변환
#    BC의 AGE_CD 1~6, GENDER_CD 1(남)·2(여) 체계에 맞춘다
long["AGE_CD"] = long["연령대"].map({
    "20대이하": "1", "20대": "2", "30대": "3",
    "40대": "4", "50대": "5", "60대이상": "6"
})
long["GENDER_CD"] = long["성별"].map({"남": "1", "여": "2"})

# 3) 11구간을 6구간으로 합산
#    as_index=False : groupby 결과를 인덱스 대신 평범한 컬럼으로 받는다
pop_long = long.groupby(
    ["행정구역명", "행정구역코드", "STRD_YYMM", "GENDER_CD", "AGE_CD"],
    as_index=False
)["인구"].sum()

print("결과:", pop_long.shape, "(285지역 × 6개월 × 2성별 × 6연령 = 20,520행)")
print()
print(pop_long.head(6).to_string())


In [ ]:
# ===== 지역 단위 정리 + 화성시 1월 보정 (pop_long에서 항상 새로 시작) =====

# [1] 시도 합계 행 제거 — 행정구역코드 3~5번째 자리가 "000"이면 시도
#     세종은 3600000000(시도) / 3611000000(시군구) 두 행 → 시군구 쪽만 자동으로 남는다
sigungu = pop_long[pop_long["행정구역코드"].str[2:5] != "000"].copy()

# [2] 분구 도시의 상위 시 행 제거 (수원시 vs 수원시 장안구 이중 계산 방지)
sigungu["단어수"] = sigungu["행정구역명"].str.split().apply(len)
분구도시 = set(" ".join(x.split()[:2])
             for x in sigungu.loc[sigungu["단어수"] == 3, "행정구역명"])
leaf = sigungu[~((sigungu["단어수"] == 2) &
                 (sigungu["행정구역명"].isin(분구도시)))].copy()

# 보정 전 상태를 그래프용으로 저장 (1월에 화성시 4개 구가 0인 상태)
보정전 = leaf.groupby("STRD_YYMM")["인구"].sum()

# [3] 화성시 1월 배분 — 성별×연령 12개 조합마다 따로
키 = ["GENDER_CD", "AGE_CD"]

feb_화성 = leaf[(leaf["STRD_YYMM"] == "202602") &
               (leaf["행정구역명"].str.startswith("경기도 화성시 "))]
jan_화성 = pop_long[(pop_long["STRD_YYMM"] == "202601") &
                   (pop_long["행정구역명"] == "경기도 화성시")]

구합 = feb_화성.groupby(키)["인구"].sum().rename("구합")
배분표 = (feb_화성
        .merge(구합, on=키)
        .merge(jan_화성[키 + ["인구"]].rename(columns={"인구": "시전체"}), on=키))
배분표["인구"] = (배분표["인구"] / 배분표["구합"] * 배분표["시전체"]).round().astype(int)
배분표["STRD_YYMM"] = "202601"     # 2월 행에서 출발했으므로 기준월을 1월로 교체

# [4] 1월 화성시 구 행(값 0)을 버리고 배분값으로 교체
버릴행 = ((leaf["STRD_YYMM"] == "202601") &
        (leaf["행정구역명"].str.startswith("경기도 화성시 ")))
leaf = pd.concat([
    leaf[~버릴행],
    배분표[["행정구역명", "행정구역코드", "STRD_YYMM", "GENDER_CD", "AGE_CD", "인구", "단어수"]]
], ignore_index=True)

# [5] 검증
print("행 수:", len(leaf), "(255 × 6 × 2 × 6 = 18,360)")
print("지역 수:", leaf["행정구역명"].nunique())
print()
print("보정 전 → 보정 후")
for 월 in sorted(보정전.index):
    print(f"  {월}: {보정전[월]:>11,} → {leaf.groupby('STRD_YYMM')['인구'].sum()[월]:>11,}")


In [ ]:
import matplotlib.pyplot as plt

# 윈도우에서 한글이 네모(□)로 깨지는 걸 막는 설정 — 그래프 그리기 전에 항상 필요
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False   # 음수 부호(−)도 깨지므로 같이 처리

# 색상은 역할별로 고정 (색맹 안전성이 검증된 조합)
파랑, 주황 = "#2a78d6", "#eb6834"                    # 보정 후 / 보정 전
잉크, 보조잉크, 흐린잉크 = "#0b0b0b", "#52514e", "#898781"   # 제목 / 설명 / 축
격자, 축선, 바탕 = "#e1e0d9", "#c3c2b7", "#fcfcfb"

# 만 명 단위로 바꿔서 축 숫자를 읽기 쉽게 만든다
전 = (보정전.sort_index() / 10000).tolist()
후 = (leaf.groupby("STRD_YYMM")["인구"].sum().sort_index() / 10000).tolist()
x = range(6)

fig, ax = plt.subplots(figsize=(9, 4.6), facecolor=바탕)
ax.set_facecolor(바탕)

# 보정 후를 먼저(아래), 보정 전을 점선으로 위에 그린다
# → 2~6월은 두 값이 같아서 선이 완전히 겹치는데, 점선이 위에 있어야 겹침이 보인다
ax.plot(x, 후, color=파랑, lw=2.5, marker="o", ms=7, mfc=바탕, mew=2,
        label="보정 후", zorder=3)
ax.plot(x, 전, color=주황, lw=2, ls="--", marker="o", ms=7, mfc=바탕, mew=2,
        label="보정 전", zorder=4)

# 1월의 격차를 양방향 화살표로 강조
ax.annotate("", xy=(0, 후[0]), xytext=(0, 전[0]),
            arrowprops=dict(arrowstyle="<->", color=흐린잉크, lw=1.2))

# 값을 직접 표기 (범례만 있으면 어느 선이 얼마인지 눈이 왔다갔다 한다)
ax.text(0.12, 후[0] + 3, f"보정 후 {후[0]:,.0f}만", color=파랑,
        fontsize=10, fontweight="bold", va="bottom")
ax.text(0.12, 전[0] - 3, f"보정 전 {전[0]:,.0f}만", color=주황,
        fontsize=10, fontweight="bold", va="top")

# 원인과 처리 방법을 그래프 안에 적어 둔다 (발표 때 따로 설명이 필요 없게)
ax.text(2.3, 5045,
        "화성시 동탄·만세·병점·효행구는 2026년 2월 신설\n"
        "→ 주민등록 통계상 1월 인구가 0으로 기록됨\n"
        "→ 2월 구별 성별·연령 구성비로 약 99만 명 배분",
        color=보조잉크, fontsize=10, va="center", linespacing=1.6)

ax.text(5.9, 5117, "2~6월은 두 계열 값이 동일", color=흐린잉크,
        fontsize=9, va="bottom", ha="right", style="italic")

ax.set_title("전처리 보정: 화성시 분구 신설로 인한 2026년 1월 인구 결측",
             color=잉크, fontsize=13, fontweight="bold", pad=14, loc="left")
ax.set_ylabel("전국 인구 (만 명)", color=보조잉크, fontsize=10)
ax.set_xticks(list(x))
ax.set_xticklabels(["1월", "2월", "3월", "4월", "5월", "6월"])
ax.set_xlim(-0.4, 5.95)
ax.set_ylim(4995, 5130)          # 99만 명 차이가 보이도록 0부터 시작하지 않음

ax.tick_params(colors=흐린잉크, labelsize=10, length=0)
ax.grid(axis="y", color=격자, lw=1)   # 가로선만, 흐리게 → 데이터가 주인공
ax.set_axisbelow(True)                # 격자를 선 뒤로 보낸다
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)   # 위·오른쪽 테두리 제거
for s in ["left", "bottom"]:
    ax.spines[s].set_color(축선)
ax.legend(frameon=False, loc="lower right", fontsize=10, labelcolor=보조잉크, ncol=2)

plt.tight_layout()
plt.show()


In [ ]:
# 저장 전 정리 — 분석에 필요한 컬럼만 남긴다 ('단어수'는 필터용 임시 컬럼이라 버림)
pop_final = leaf[["행정구역명", "행정구역코드", "STRD_YYMM",
                  "GENDER_CD", "AGE_CD", "인구"]].copy()

# 보기 좋게 정렬 (지역 → 월 → 성별 → 연령 순)
pop_final = pop_final.sort_values(
    ["행정구역명", "STRD_YYMM", "GENDER_CD", "AGE_CD"]
).reset_index(drop=True)

# 최종 점검
print("행 수:", len(pop_final), "(255 × 6 × 2 × 6 = 18,360)")
print("지역 수:", pop_final["행정구역명"].nunique())
print("결측:", pop_final.isna().sum().sum(), "건")
print("인구 0인 행:", (pop_final["인구"] == 0).sum(), "건")
print()
print(pop_final.head(12).to_string())

# encoding="utf-8-sig" : 엑셀에서 열었을 때 한글이 깨지지 않게 BOM을 붙인다
#                        (그냥 utf-8로 저장하면 엑셀이 한글을 깨뜨린다)
pop_final.to_csv("data/인구_시군구_성별_연령_202601_202606.csv",
                 index=False, encoding="utf-8-sig")

print()
print("저장 완료 → data/인구_시군구_성별_연령_202601_202606.csv")
